# EDA ДТП - Амурская область

Рабочий notebook для исследования данных перед переносом результатов в `solution/app.py` и `solution/analysis.md`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "vscode"
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from solution.data_loading import load_records, records_to_frame

DATA_PATH = ROOT / 'data' / 'raw' / 'amurskaia-oblast.geojson'
records = load_records(DATA_PATH)
df = records_to_frame(records)
df.head()

## 1. Размер, период и структура данных

In [ ]:
print('Размер таблицы:', df.shape)
print('Период:', df['datetime'].min(), '-', df['datetime'].max())
print('Уникальных ДТП:', df['id'].nunique())
print('Дубликатов ID:', df.loc[df['id'].notna(), 'id'].duplicated().sum())
df.dtypes.to_frame('dtype')

In [ ]:
summary = pd.DataFrame({
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique': df.nunique(dropna=True),
})
summary

## 2. Категориальные признаки

In [ ]:
for column in ['region', 'category', 'severity', 'light']:
    print(f'\n--- {column} ---')
    display(df[column].value_counts(dropna=False).to_frame('count'))

## 3. Числовые признаки и необычные значения

In [ ]:
numeric = ['participants_count', 'injured_count', 'dead_count']
display(df[numeric].describe().T)

for column in numeric:
    display(df.nlargest(10, column)[['id', 'datetime', 'region', 'severity', column]])

## 4. Временная динамика

In [ ]:
monthly = (df.set_index('datetime').resample('MS').size()
          .rename('crashes').reset_index())
fig = px.line(monthly, x='datetime', y='crashes', markers=True,
              title='Количество ДТП по месяцам',
              labels={'datetime': 'Месяц', 'crashes': 'Количество ДТП'})
fig.show()

In [ ]:
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour
fig = px.histogram(df, x='hour', nbins=24, title='Распределение ДТП по часам',
                   labels={'hour': 'Час', 'count': 'Количество ДТП'})
fig.show()

## 5. Сравнение территорий и последствий

In [ ]:
region_counts = df['region'].value_counts().rename_axis('region').reset_index(name='crashes')
fig = px.bar(region_counts.head(20), x='crashes', y='region', orientation='h',
             title='Территории с наибольшим числом ДТП',
             labels={'crashes': 'Количество ДТП', 'region': 'Территория'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
severity_by_light = pd.crosstab(df['light'], df['severity'], normalize='index').mul(100).round(1)
severity_by_light

In [ ]:
fig = px.bar(severity_by_light.reset_index(), x='light', y=severity_by_light.columns,
             barmode='group', title='Структура тяжести ДТП по условиям освещения',
             labels={'light': 'Освещение', 'value': 'Доля, %', 'variable': 'Тяжесть'})
fig.show()

## 6. Координаты и карта

Проверяем долю пригодных координат. Для анализа карт используем только `map_valid == True`.

In [ ]:
print('Всего записей:', len(df))
print('Пригодны для карты:', int(df['map_valid'].sum()))
print('Непригодны для карты:', int((~df['map_valid']).sum()))
points = df[df['map_valid']].copy()
fig = px.scatter_map(points.sample(min(len(points), 5000), random_state=42),
                     lat='latitude', lon='longitude', color='severity',
                     hover_data=['datetime', 'region', 'category'],
                     zoom=5, height=600, title='Точечная карта ДТП',
                     map_style='open-street-map')
fig.show()

## 7. Пошаговая проверка качества данных

Исходная таблица `df` не изменяется. Сначала фиксируем проблемы, затем строим отдельные выборки для разных задач.

In [ ]:
df_raw = df.copy()

passport = pd.DataFrame({
    'rows': [len(df_raw)],
    'columns': [df_raw.shape[1]],
    'date_min': [df_raw['datetime'].min()],
    'date_max': [df_raw['datetime'].max()],
    'unique_ids': [df_raw['id'].nunique()],
    'duplicate_ids': [df_raw['id'].duplicated().sum()],
    'missing_dates': [df_raw['datetime'].isna().sum()],
})
passport.T.rename(columns={0: 'value'})

### 7.1 Логические проверки

Невозможные значения определяем правилами предметной области. Большое, но возможное значение не удаляем автоматически.

In [ ]:
checks = pd.Series({
    'negative_participants': (df_raw['participants_count'] < 0).sum(),
    'negative_injured': (df_raw['injured_count'] < 0).sum(),
    'negative_dead': (df_raw['dead_count'] < 0).sum(),
    'injured_gt_participants': (df_raw['injured_count'] > df_raw['participants_count']).sum(),
    'dead_gt_participants': (df_raw['dead_count'] > df_raw['participants_count']).sum(),
    'dead_gt_injured': (df_raw['dead_count'] > df_raw['injured_count']).sum(),
    'missing_coordinates': df_raw[['longitude', 'latitude']].isna().any(axis=1).sum(),
    'invalid_for_map': (~df_raw['map_valid']).sum(),
})
checks.to_frame('count')

In [ ]:
# Список самых крупных происшествий для ручной проверки, а не для автоматического удаления
top_participants = df_raw.nlargest(10, 'participants_count')[
    ['id', 'datetime', 'region', 'category', 'severity',
     'participants_count', 'injured_count', 'dead_count']
].copy()
top_participants

### 7.2 Первичная визуальная проверка сырых значений

In [ ]:
fig = px.histogram(df_raw, x='participants_count', nbins=30,
                   title='Сырое распределение числа участников',
                   labels={'participants_count': 'Участников', 'count': 'Число ДТП'})
fig.show()

fig = px.box(df_raw, y='participants_count', points='all',
             title='Выбросы числа участников',
             labels={'participants_count': 'Участников'})
fig.show()

In [ ]:
# Сезонность: сначала считаем весь период, затем отдельно проверяем полные годы
month_counts = df_raw['datetime'].dt.month.value_counts().sort_index()
month_counts.index = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн',
                     'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']
month_counts.to_frame('crashes')

In [ ]:
complete_for_heatmap = df_raw[df_raw['datetime'].dt.year < 2026].copy()
year_month = (complete_for_heatmap.assign(year=complete_for_heatmap['datetime'].dt.year,
                                           month=complete_for_heatmap['datetime'].dt.month)
              .pivot_table(index='year', columns='month', values='id',
                           aggfunc='size', fill_value=0))
year_month = year_month.reindex(columns=range(1, 13), fill_value=0)
year_month.columns = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн',
                      'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']
fig = px.imshow(
    year_month, text_auto=True, aspect='auto', color_continuous_scale='YlOrRd',
    title='Число ДТП по месяцам и полным годам',
    labels={'x': 'Месяц', 'y': 'Год', 'color': 'Число ДТП'},
)
fig.update_layout(template='plotly_white', height=520)
fig.show()

### 7.3 Создание выборок после проверки

Одна таблица не подходит для всех задач: для карты нужны валидные координаты, а для межгодовых сравнений - полные годы.

In [ ]:
df_all = df_raw.copy()
df_map = df_raw[df_raw['map_valid']].copy()
df_valid_counts = df_raw[
    (df_raw['participants_count'] >= 0)
    & (df_raw['injured_count'] >= 0)
    & (df_raw['dead_count'] >= 0)
].copy()

# 2026 неполный, поэтому исключаем его только для сравнений между годами
df_complete_years = df_raw[df_raw['datetime'].dt.year < 2026].copy()

pd.Series({
    'all_records': len(df_all),
    'map_records': len(df_map),
    'valid_count_records': len(df_valid_counts),
    'complete_year_records': len(df_complete_years),
})

In [ ]:
# Анализ чувствительности: сравниваем показатели с крупнейшим ДТП и без него
max_participants_id = df_raw.loc[df_raw['participants_count'].idxmax(), 'id']
without_extreme = df_raw[df_raw['id'] != max_participants_id]
sensitivity = pd.DataFrame({
    'with_52_participants': [df_raw['participants_count'].mean(), df_raw['participants_count'].median()],
    'without_52_participants': [without_extreme['participants_count'].mean(), without_extreme['participants_count'].median()],
}, index=['mean', 'median'])
sensitivity

## 8. Основные графики для исследования

Все графики ниже используют одну запись как одно ДТП. Для межгодовых сравнений исключён неполный 2026 год; для карт - записи с ошибочными координатами.

In [ ]:
# Рабочие выборки для разных задач
AMUR_BOUNDS = {'lon_min': 119, 'lon_max': 135, 'lat_min': 48, 'lat_max': 58}

df_all = df.copy()
df_complete_years = df_all[df_all['datetime'].dt.year < 2026].copy()
df_map = df_all[
    df_all['map_valid']
    & df_all['longitude'].between(AMUR_BOUNDS['lon_min'], AMUR_BOUNDS['lon_max'])
    & df_all['latitude'].between(AMUR_BOUNDS['lat_min'], AMUR_BOUNDS['lat_max'])
].copy()

print(f'Для карт: {len(df_map):,} из {len(df_all):,} ДТП; исключено: {len(df_all) - len(df_map):,}.')

### 8.1 Динамика во времени

In [ ]:
# 1. Сравниваем только полные годы - 2026 здесь нет
yearly = (df_complete_years.groupby(df_complete_years['datetime'].dt.year)
          .size().rename('ДТП').reset_index().rename(columns={'datetime': 'Год'}))
fig = px.bar(yearly, x='Год', y='ДТП', text='ДТП',
             title='Число ДТП по полным годам',
             labels={'Год': 'Год', 'ДТП': 'Число ДТП'})
fig.update_traces(textposition='outside')
fig.show()

# 2. Сезонность: среднее число ДТП за месяц среди полных лет
month_names = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']
monthly_mean = (df_complete_years.assign(месяц=df_complete_years['datetime'].dt.month)
                .groupby('месяц').size().div(df_complete_years['datetime'].dt.year.nunique())
                .reindex(range(1, 13), fill_value=0).rename('Среднее число ДТП').reset_index())
monthly_mean['Название месяца'] = month_names
fig = px.line(monthly_mean, x='Название месяца', y='Среднее число ДТП', markers=True,
              title='Среднее число ДТП по месяцам (2015–2025)')
fig.show()

# 3. Распределение ДТП по часам
hours = (df_all.assign(Час=df_all['datetime'].dt.hour).groupby('Час').size()
         .reindex(range(24), fill_value=0).rename('ДТП').reset_index())
fig = px.bar(hours, x='Час', y='ДТП', title='Распределение ДТП по часам суток',
             labels={'Час': 'Час начала ДТП', 'ДТП': 'Число ДТП'})
fig.update_xaxes(dtick=1)
fig.show()

1) Среди полных лет максимум зарегистрирован в 2015 году (1390 ДТП), минимум - в 2021 (1069); монотонного тренда нет.
2) Максимальное среднее число ДТП приходится на июль (~140), минимальное - на февраль (~60); 2026 год исключён как неполный.
3) Максимум по часам приходится на 18 (~900 ДТП), минимум - на 5 (~150). Возможно это связано с интенсивностью движения, но данных о потоке нет.

### 8.2 Типы ДТП, тяжесть и территории

In [ ]:
# 4. Наиболее частые типы ДТП
categories = df_all['category'].value_counts().rename_axis('Тип ДТП').reset_index(name='ДТП')
fig = px.bar(categories, x='ДТП', y='Тип ДТП', orientation='h',
             title='Типы ДТП', labels={'ДТП': 'Число ДТП'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=650)
fig.show()

# 5. Распределение тяжести последствий
severity = df_all['severity'].value_counts().rename_axis('Тяжесть').reset_index(name='ДТП')
fig = px.bar(severity, x='Тяжесть', y='ДТП', text='ДТП', color='Тяжесть',
             title='Тяжесть последствий ДТП', labels={'ДТП': 'Число ДТП'})
fig.update_traces(textposition='outside')
fig.show()

# 6. В каждой группе освещения доли суммируются до 100 %
severity_light = (pd.crosstab(df_all['light'], df_all['severity'], normalize='index')
                  .mul(100).reset_index().melt(id_vars='light', var_name='Тяжесть', value_name='Доля, %'))
fig = px.bar(severity_light, x='light', y='Доля, %', color='Тяжесть', barmode='stack',
             title='Структура тяжести ДТП при разном освещении',
             labels={'light': 'Условия освещения'})
fig.update_xaxes(tickangle=-25)
fig.show()

# 7. Территории с наибольшим числом ДТП
regions = df_all['region'].value_counts().head(15).sort_values().rename_axis('Территория').reset_index(name='ДТП')
fig = px.bar(regions, x='ДТП', y='Территория', orientation='h',
             title='15 территорий с наибольшим числом ДТП')
fig.show()

1) ДТП с погибшими - 1205: примерно в 5 раза меньше лёгких и тяжёлых ДТП.
2) Доля ДТП с погибшими ночью без освещения около 20%, а днём и ночью при включённом освещении - около 7%;.
3) В Благовещенске зарегистрировано ~4500 ДТП. Возможно это связано с Населенностью города и потоком.

### 8.3 Карты: точки и агрегированные гексагоны

Для агрегации координаты переводятся в метрическую проекцию, после чего строятся одинаковые гексагоны радиусом 1 км. Цвет показывает число ДТП внутри гексагона.

In [ ]:
# 8. Точечная карта: подвыборка нужна только для скорости отрисовки
points = df_map.sample(min(len(df_map), 5_000), random_state=42)
fig = px.scatter_map(points, lat='latitude', lon='longitude', color='severity',
                     hover_data=['datetime', 'region', 'category'],
                     center={'lat': df_map['latitude'].median(), 'lon': df_map['longitude'].median()},
                     zoom=5, height=600, opacity=0.55, map_style='open-street-map',
                     title='Точечная карта ДТП (случайная подвыборка до 5 000 точек)')
fig.show()

# 9. Агрегированная карта: число ДТП в гексагонах радиусом 1 км
import json
import geopandas as gpd
from shapely.geometry import Polygon

HEX_RADIUS_M = 1_000
MAP_CRS = 'EPSG:3576'  # метрическая равновеликая проекция для территории России
points_gdf = gpd.GeoDataFrame(
    df_map[['id', 'severity']].copy(),
    geometry=gpd.points_from_xy(df_map['longitude'], df_map['latitude']),
    crs='EPSG:4326',
).to_crs(MAP_CRS)

min_x, min_y, max_x, max_y = points_gdf.total_bounds
x_step = 1.5 * HEX_RADIUS_M
y_step = np.sqrt(3) * HEX_RADIUS_M
angles = np.radians(np.arange(0, 360, 60))
hexagons = []
for column, center_x in enumerate(np.arange(min_x - HEX_RADIUS_M, max_x + HEX_RADIUS_M, x_step)):
    y_offset = (column % 2) * y_step / 2
    for center_y in np.arange(min_y - HEX_RADIUS_M + y_offset, max_y + HEX_RADIUS_M, y_step):
        vertices = [
            (center_x + HEX_RADIUS_M * np.cos(angle),
             center_y + HEX_RADIUS_M * np.sin(angle))
            for angle in angles
        ]
        hexagons.append(Polygon(vertices))

hex_grid = gpd.GeoDataFrame(geometry=hexagons, crs=MAP_CRS)
point_hex = gpd.sjoin(points_gdf, hex_grid, how='inner', predicate='within')
hex_summary = point_hex.groupby('index_right').agg(
    ДТП=('id', 'size'),
    ДТП_с_погибшими=('severity', lambda values: (values == 'С погибшими').sum()),
)
hex_map = hex_grid.join(hex_summary, how='inner').to_crs('EPSG:4326').reset_index(drop=True)
hex_geojson = json.loads(hex_map.to_json())

fig = px.choropleth_map(
    hex_map, geojson=hex_geojson, locations=hex_map.index.astype(str), featureidkey='id',
    color='ДТП', hover_data=['ДТП_с_погибшими'], color_continuous_scale='YlOrRd',
    center={'lat': float(df_map['latitude'].median()), 'lon': float(df_map['longitude'].median())},
    zoom=5, height=600, map_style='open-street-map', opacity=0.72,
    title='Концентрация ДТП по гексагонам радиусом 1 км',
    labels={'ДТП': 'Число ДТП', 'ДТП_с_погибшими': 'С погибшими'},
)
fig.show()
print(f'В гексагоны попало {len(point_hex):,} из {len(df_map):,} ДТП; заполнено гексагонов: {len(hex_map):,}.')

## 9. Исследовательский вопрос: объясняет ли тип ДТП связь освещения и тяжести?

Сравниваем только ночные ДТП с включённым и отсутствующим освещением. Категория «освещение не включено» не входит в сравнение: это отдельное, малочисленное условие.

In [ ]:
# В трёх наиболее частых типах достаточно наблюдений для содержательного сравнения
TOP_CATEGORIES = ['Столкновение', 'Наезд на пешехода', 'Съезд с дороги']
LIGHT_GROUPS = {
    'В темное время суток, освещение включено': 'Освещение включено',
    'В темное время суток, освещение отсутствует': 'Освещение отсутствует',
}
night_light = df_all[
    df_all['category'].isin(TOP_CATEGORIES) & df_all['light'].isin(LIGHT_GROUPS)
].assign(
    Освещение=lambda x: x['light'].map(LIGHT_GROUPS),
    С_погибшими=lambda x: x['severity'].eq('С погибшими'),
)

fatal_by_type_light = (night_light.groupby(['category', 'Освещение'], observed=True)
                       .agg(ДТП=('id', 'size'), ДТП_с_погибшими=('С_погибшими', 'sum'))
                       .reset_index())
fatal_by_type_light['Доля ДТП с погибшими, %'] = (
    100 * fatal_by_type_light['ДТП_с_погибшими'] / fatal_by_type_light['ДТП']
).round(1)
fatal_by_type_light

In [ ]:
fig = px.bar(
    fatal_by_type_light, x='category', y='Доля ДТП с погибшими, %', color='Освещение',
    barmode='group', text='Доля ДТП с погибшими, %',
    category_orders={'category': TOP_CATEGORIES},
    title='Доля ДТП с погибшими ночью: сравнение внутри типа ДТП',
    labels={'category': 'Тип ДТП'},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, fatal_by_type_light['Доля ДТП с погибшими, %'].max() + 4])
fig.show()

Можно сделать вывод о том, что при отсутствие освещения доля ДТП с погибшими увеличивается, за исключением Съезда с дороги. Это можно объяснить, что Съезд с дороги был вызван не из-за плохого освещения(например гололед)

In [ ]:
runoff_complete = df_complete_years[df_complete_years['category'].eq('Съезд с дороги')].copy()
runoff_complete['Период'] = np.where(
    runoff_complete['datetime'].dt.month.isin([12, 1, 2]),
    'Зима (дек–фев)', 'Не зима (мар–ноя)',
)
months_in_period = {'Зима (дек–фев)': 11 * 3, 'Не зима (мар–ноя)': 11 * 9}
runoff_season = runoff_complete['Период'].value_counts().rename_axis('Период').reset_index(name='Всего съездов')
runoff_season['Число месяцев'] = runoff_season['Период'].map(months_in_period)
runoff_season['Среднее съездов в месяц'] = (
    runoff_season['Всего съездов'] / runoff_season['Число месяцев']
).round(1)

fig = px.bar(
    runoff_season, x='Период', y='Среднее съездов в месяц',
    text='Среднее съездов в месяц', color='Период',
    hover_data=['Всего съездов', 'Число месяцев'],
    title='Съезды с дороги зимой и в остальные месяцы (2015–2025)',
    color_discrete_map={'Зима (дек–фев)': '#5B8DB8', 'Не зима (мар–ноя)': '#D18B32'},
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(template='plotly_white', showlegend=False, height=440)
fig.show()
runoff_season

Мое предположение оказалось ложным - гололед не основная причина съезда с дороги

## 10. Исследовательские вопросы и три гипотезы

### Исследовательские вопросы

1. Как меняется число ДТП по годам, месяцам и часам суток?
2. Связано ли отсутствие ночного освещения с долей ДТП с погибшими при наездах на пешеходов?
3. Различается ли тяжесть столкновений и съездов с дороги?
4. Чем состав ДТП в Благовещенске отличается от остальной области?
5. Какие территории имеют похожие профили ДТП? Этот вопрос рассматривается с помощью обязательных по ТЗ PCA и t-SNE.

Во всех проверках α = 0,05. Эффект в каждой гипотезе измеряется одинаково - разностью долей в процентных пунктах. Для 95% доверительных интервалов применяется bootstrap с seed 42.

### Гипотеза 1. Освещение при наездах на пешеходов

- Группы: ночные наезды на пешеходов без освещения и при включённом освещении.
- H0: доли ДТП с погибшими в двух группах равны.
- H1: доли различаются.
- Тест: двусторонний перестановочный тест разности долей, 10 000 перестановок.

### Гипотеза 2. Столкновения и съезды с дороги

- H0: доля ДТП с погибшими одинакова для столкновений и съездов с дороги.
- H1: доли различаются.
- Тест: χ²-тест таблицы 2×2 «тип ДТП × наличие погибших».

### Гипотеза 3. Съезды с дороги в городе и области

- H0: доля съездов с дороги одинакова в Благовещенске и на остальных территориях.
- H1: доли различаются.
- Тест: χ²-тест таблицы 2×2 «территория × съезд с дороги».

Единица анализа - одно ДТП. Результаты показывают связи в выгрузке и не доказывают причины различий.

## 11. Результаты статистических проверок

Перестановочный тест случайно перемешивает принадлежность ДТП к группам и проверяет, часто ли при H0 возникает наблюдаемая разница. Bootstrap многократно пересобирает каждую группу с возвращением и даёт 95% интервал для разности долей. В обоих расчётах используется seed 67.

In [ ]:
from scipy.stats import chi2_contingency
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RANDOM_SEED = 67
ALPHA = 0.05
N_RESAMPLES = 10000

def bootstrap_difference(first, second, n_resamples=N_RESAMPLES, seed=RANDOM_SEED):
    # 95% bootstrap CI для разности долей: first minus second.
    rng = np.random.default_rng(seed)
    differences = np.empty(n_resamples)
    for index in range(n_resamples):
        differences[index] = (
            rng.choice(first, len(first), replace=True).mean()
            - rng.choice(second, len(second), replace=True).mean()
        )
    return 100 * np.quantile(differences, [0.025, 0.975])

### 11.1 Гипотеза 1: освещение при наездах на пешеходов

Сравниваем долю ДТП с погибшими среди ночных наездов на пешеходов без освещения и при включённом освещении. В перестановочном тесте метки групп перемешиваются 10000 раз.

In [ ]:
pedestrian_night = df_all[
    df_all['category'].eq('Наезд на пешехода')
    & df_all['light'].isin(LIGHT_GROUPS)
].assign(fatal=lambda x: x['severity'].eq('С погибшими'))
unlit_fatal = pedestrian_night.loc[
    pedestrian_night['light'].eq('В темное время суток, освещение отсутствует'), 'fatal'
].to_numpy()
lit_fatal = pedestrian_night.loc[
    pedestrian_night['light'].eq('В темное время суток, освещение включено'), 'fatal'
].to_numpy()
observed_difference = unlit_fatal.mean() - lit_fatal.mean()

rng = np.random.default_rng(RANDOM_SEED)
pooled = np.concatenate([unlit_fatal, lit_fatal])
permutation_differences = np.empty(N_RESAMPLES)
for index in range(N_RESAMPLES):
    shuffled = rng.permutation(pooled)
    permutation_differences[index] = (
        shuffled[:len(unlit_fatal)].mean() - shuffled[len(unlit_fatal):].mean()
    )
permutation_pvalue = (
    np.count_nonzero(np.abs(permutation_differences) >= abs(observed_difference)) + 1
) / (N_RESAMPLES + 1)
h1_ci = bootstrap_difference(unlit_fatal, lit_fatal)

h1_result = pd.DataFrame({
    'Без освещения, n': [len(unlit_fatal)],
    'При освещении, n': [len(lit_fatal)],
    'Доля без освещения, %': [100 * unlit_fatal.mean()],
    'Доля при освещении, %': [100 * lit_fatal.mean()],
    'Разность, п.п.': [100 * observed_difference],
    '95% CI': [f'[{h1_ci[0]:.1f}; {h1_ci[1]:.1f}]'],
    'p-value': [permutation_pvalue],
})
h1_result.round(4)

In [ ]:
pedestrian_plot = pd.DataFrame({
    'Освещение': ['Включено', 'Отсутствует'],
    'Доля ДТП с погибшими, %': [100 * lit_fatal.mean(), 100 * unlit_fatal.mean()],
    'Число ДТП': [len(lit_fatal), len(unlit_fatal)],
})
fig = px.bar(
    pedestrian_plot, x='Освещение', y='Доля ДТП с погибшими, %',
    text='Доля ДТП с погибшими, %', color='Освещение',
    title='Наезды на пешеходов ночью: доля ДТП с погибшими',
    hover_data=['Число ДТП'],
    color_discrete_map={'Включено': '#3579B8', 'Отсутствует': '#B33440'},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_white', showlegend=False, height=440)
fig.show()

Без освещения погибшие были в 30.5% наездов на пешеходов, при включённом освещении в 10.5%. Разность равна 20 п.п. 95% bootstrap CI [15.0; 24.9] p перестановочного теста < 0.001. H0 отклоняется - в этой выгрузке доли различаются.

### 11.2 Гипотеза 2: столкновения и съезды с дороги

In [ ]:
runoff_fatal = df_all.loc[df_all['category'].eq('Съезд с дороги'), 'severity'].eq('С погибшими').to_numpy()
collision_fatal = df_all.loc[df_all['category'].eq('Столкновение'), 'severity'].eq('С погибшими').to_numpy()
h2_table = np.array([
    [runoff_fatal.sum(), len(runoff_fatal) - runoff_fatal.sum()],
    [collision_fatal.sum(), len(collision_fatal) - collision_fatal.sum()],
])
h2_chi2, h2_pvalue, h2_df, h2_expected = chi2_contingency(h2_table)
h2_difference = 100 * (runoff_fatal.mean() - collision_fatal.mean())
h2_ci = bootstrap_difference(runoff_fatal, collision_fatal)
display(pd.DataFrame({
    'Съезды, n': [len(runoff_fatal)], 'Столкновения, n': [len(collision_fatal)],
    'Съезды с погибшими, %': [100 * runoff_fatal.mean()],
    'Столкновения с погибшими, %': [100 * collision_fatal.mean()],
    'Разность, п.п.': [h2_difference], '95% CI': [f'[{h2_ci[0]:.1f}; {h2_ci[1]:.1f}]'],
    'χ²': [h2_chi2], 'p-value': [h2_pvalue],
}).round(4))

h2_plot = pd.DataFrame({
    'Тип ДТП': ['Столкновение', 'Съезд с дороги'],
    'Доля ДТП с погибшими, %': [100 * collision_fatal.mean(), 100 * runoff_fatal.mean()],
    'Число ДТП': [len(collision_fatal), len(runoff_fatal)],
})
fig = px.bar(
    h2_plot, x='Тип ДТП', y='Доля ДТП с погибшими, %',
    text='Доля ДТП с погибшими, %', color='Тип ДТП', hover_data=['Число ДТП'],
    title='Доля ДТП с погибшими: столкновения и съезды с дороги',
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_white', showlegend=False, height=440)
fig.show()

Погибшие были в 11.4% съездов с дороги и в 7% столкновений. Разность равна 4.4 п.п. 95% bootstrap CI [2.7; 6.2]. HI^2 = 29.01 p < 0.001, поэтому H0 отклоняется - доля ДТП с погибшими для этих двух типов различается.

### 11.3 Гипотеза 3: съезды с дороги в Благовещенске и области

In [ ]:
city_runoff = df_all.loc[df_all['region'].eq('Благовещенск'), 'category'].eq('Съезд с дороги').to_numpy()
region_runoff = df_all.loc[~df_all['region'].eq('Благовещенск'), 'category'].eq('Съезд с дороги').to_numpy()
h3_table = np.array([
    [region_runoff.sum(), len(region_runoff) - region_runoff.sum()],
    [city_runoff.sum(), len(city_runoff) - city_runoff.sum()],
])
h3_chi2, h3_pvalue, h3_df, h3_expected = chi2_contingency(h3_table)
h3_difference = 100 * (region_runoff.mean() - city_runoff.mean())
h3_ci = bootstrap_difference(region_runoff, city_runoff)
display(pd.DataFrame({
    'Остальная область, n': [len(region_runoff)], 'Благовещенск, n': [len(city_runoff)],
    'Съезды в области, %': [100 * region_runoff.mean()],
    'Съезды в Благовещенске, %': [100 * city_runoff.mean()],
    'Разность, п.п.': [h3_difference], '95% CI': [f'[{h3_ci[0]:.1f}; {h3_ci[1]:.1f}]'],
    'χ²': [h3_chi2], 'p-value': [h3_pvalue],
}).round(4))

h3_plot = pd.DataFrame({
    'Территория': ['Благовещенск', 'Остальная область'],
    'Доля съездов с дороги, %': [100 * city_runoff.mean(), 100 * region_runoff.mean()],
    'Число всех ДТП': [len(city_runoff), len(region_runoff)],
})
fig = px.bar(
    h3_plot, x='Территория', y='Доля съездов с дороги, %',
    text='Доля съездов с дороги, %', color='Территория', hover_data=['Число всех ДТП'],
    title='Доля съездов с дороги: Благовещенск и остальная область',
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_white', showlegend=False, height=440)
fig.show()

Съезды с дороги составляют 15.8% в остальной области и 1.7% в Благовещенске. Разность равна 14.1 п.п., 95% bootstrap CI [13.1; 14.9]. HI^2 = 591.30, p < 0.001, поэтому H0 отклоняется: доли заметно различаются. Без данных о дорогах и транспортном потоке этот результат не объясняет причину.

## 12. PCA и t-SNE: профили территорий

Объект анализа - одна из 24 территорий. Признаки: логарифм числа ДТП, доли ДТП с погибшими и тяжёлыми последствиями, доли основных типов ДТП, доля ночных ДТП без освещения и среднее число участников. Доли позволяют сравнивать структуру, а не только размер территории. Все признаки стандартизируются.

In [ ]:
unlit_value = 'В темное время суток, освещение отсутствует'
territory_profiles = pd.DataFrame(index=sorted(df_all['region'].unique()))
grouped_region = df_all.groupby('region')
territory_profiles['log_n'] = np.log1p(grouped_region.size())
territory_profiles['fatal_share'] = df_all.assign(x=df_all['severity'].eq('С погибшими')).groupby('region')['x'].mean()
territory_profiles['heavy_share'] = df_all.assign(x=df_all['severity'].eq('Тяжёлый')).groupby('region')['x'].mean()
territory_profiles['pedestrian_share'] = df_all.assign(x=df_all['category'].eq('Наезд на пешехода')).groupby('region')['x'].mean()
territory_profiles['collision_share'] = df_all.assign(x=df_all['category'].eq('Столкновение')).groupby('region')['x'].mean()
territory_profiles['runoff_share'] = df_all.assign(x=df_all['category'].eq('Съезд с дороги')).groupby('region')['x'].mean()
territory_profiles['rollover_share'] = df_all.assign(x=df_all['category'].eq('Опрокидывание')).groupby('region')['x'].mean()
territory_profiles['night_unlit_share'] = df_all.assign(x=df_all['light'].eq(unlit_value)).groupby('region')['x'].mean()
territory_profiles['mean_participants'] = grouped_region['participants_count'].mean()

scaler = StandardScaler()
profile_scaled = scaler.fit_transform(territory_profiles)
pca_model = PCA(n_components=2)
pca_xy = pca_model.fit_transform(profile_scaled)
pca_frame = pd.DataFrame(pca_xy, columns=['PC1', 'PC2'])
pca_frame['Территория'] = territory_profiles.index
pca_frame['Число ДТП'] = grouped_region.size().reindex(territory_profiles.index).to_numpy()
pca_frame['Доля с погибшими, %'] = 100 * territory_profiles['fatal_share'].to_numpy()
loadings = pd.DataFrame(
    pca_model.components_.T, index=territory_profiles.columns, columns=['PC1', 'PC2']
)
print('Доля дисперсии PC1 и PC2:', pca_model.explained_variance_ratio_.round(3))
print('Всего сохранено:', round(pca_model.explained_variance_ratio_.sum(), 3))
display(loadings.round(3))

fig = px.scatter(
    pca_frame, x='PC1', y='PC2', size='Число ДТП', color='Доля с погибшими, %',
    hover_name='Территория', color_continuous_scale='YlOrRd', size_max=35,
    title='PCA профилей территорий: 72,3% дисперсии в двух компонентах',
)
fig.add_hline(y=0, line_width=1, line_color='#D1D5DB')
fig.add_vline(x=0, line_width=1, line_color='#D1D5DB')
fig.update_layout(template='plotly_white', height=600)
fig.show()

C1 объясняет около 50,0% дисперсии и отделяет территории с большей долей съездов, опрокидываний и ночных ДТП без освещения от территорий, где выше доли столкновений и наездов на пешеходов. PC2 объясняет 22,4% и сильнее связан с долей ДТП с погибшими и средним числом участников.

In [ ]:
tsne_results = {}
for perplexity in [5, 10]:
    tsne_results[perplexity] = TSNE(
        n_components=2, perplexity=perplexity, random_state=RANDOM_SEED,
        init='pca', learning_rate='auto', max_iter=1000,
    ).fit_transform(profile_scaled)

fig = make_subplots(rows=1, cols=2, subplot_titles=['perplexity = 5', 'perplexity = 10'])
for column, perplexity in enumerate([5, 10], start=1):
    coords = tsne_results[perplexity]
    fig.add_trace(go.Scatter(
        x=coords[:, 0], y=coords[:, 1], mode='markers',
        marker={
            'size': 11, 'color': pca_frame['Доля с погибшими, %'],
            'colorscale': 'YlOrRd', 'showscale': column == 2,
            'colorbar': {'title': 'С погибшими, %'},
        },
        text=territory_profiles.index,
        hovertemplate='%{text}<extra></extra>', showlegend=False,
    ), row=1, col=column)
fig.update_layout(
    title='t-SNE профилей одних и тех же 24 территорий, seed = 42',
    template='plotly_white', height=560,
)
fig.update_xaxes(title_text='t-SNE 1')
fig.update_yaxes(title_text='t-SNE 2')
fig.show()

Локальные соседства, которые сохраняются в обоих запусках, можно считать более устойчивыми, чем форму облака или расстояния между далёкими группами. можно сказть, что похожесть устойчива, ведь она не меняется при разных perplexity

## 13. Главные результаты и ограничения

1. **Выраженная сезонность.** В 2015–2025 годах июль был выше февраля во всех 11 годах; в среднем 137.6 против 59.7 ДТП в месяц. Это числа зарегистрированных ДТП, а не риск на одну поездку.
2. **Освещение при наездах на пешеходов.** Ночью без освещения доля ДТП с погибшими составила 30.5%, при включённом освещении - 10.5%; разность 19,9 п.п.
3. **Город и остальная область различаются по сценарию ДТП.** Съезды с дороги составляют 15.8% ДТП вне Благовещенска и 1.7% в городе.

Ограничения: выгрузка содержит только зарегистрированные ДТП; нет интенсивности движения, числа поездок, населения в зоне воздействия и точных данных о дорожной инфраструктуре. Поэтому нельзя оценить индивидуальный риск или причинный эффект освещения. В межгодовых сравнениях исключён неполный 2026 год. Для карт исключены 266 записей с отсутствующими, нулевыми либо находящимися вне приближённой рамки региона координатами.

## 14. График до и после

У исходной вертикальной диаграммы 17 длинных подписей, поэтому категории трудно сравнивать. В улучшенной версии восемь частых категорий показаны горизонтально, редкие объединены в «Прочие», добавлены числа и доли от всех 13040 ДТП.

In [ ]:
# ДО: длинные подписи конкурируют за место, редкие категории почти не видны
category_before = df_all['category'].value_counts().rename_axis('Тип ДТП').reset_index(name='ДТП')
fig_before = px.bar(
    category_before, x='Тип ДТП', y='ДТП', title='До: все типы ДТП на вертикальной оси',
)
fig_before.update_layout(template='plotly_white', height=520)
fig_before.show()

# ПОСЛЕ: редкие типы объединены, сортировка и подписи облегчают сравнение
main_types = df_all['category'].value_counts().head(8).index
category_after = (df_all.assign(Тип=np.where(df_all['category'].isin(main_types), df_all['category'], 'Прочие'))
                  ['Тип'].value_counts().sort_values().rename_axis('Тип ДТП').reset_index(name='ДТП'))
category_after['Доля, %'] = (100 * category_after['ДТП'] / len(df_all)).round(1)
category_after['Подпись'] = category_after.apply(
    lambda row: f"{row['ДТП']:,} ({row['Доля, %']:.1f}%)", axis=1
)
fig_after = px.bar(
    category_after, x='ДТП', y='Тип ДТП', orientation='h', text='Подпись',
    title='После: основные типы ДТП, число и доля от всей выборки',
    color='ДТП', color_continuous_scale=['#B9D7EA', '#2E6F9E'],
)
fig_after.update_traces(textposition='outside')
fig_after.update_layout(template='plotly_white', height=520, coloraxis_showscale=False)
fig_after.show()